# Drone Delivery — Deep RL Game (DQN)

Option 2 — Drone Delivery. A small 2D game (two fixed obstacles, one delivery
zone); a DQN agent is trained from scratch. **Runtime → Run all** executes the
whole pipeline: training (~minutes on CPU), 10-episode evaluation, and the
plot. A pre-trained policy (`dqn_drone.pth`) is included; if the checkpoint is
present you may skip the training cell.

Files: this notebook, `drone_env.py`, `dqn_agent.py`, `train.py`, `evaluate.py`,
`play.py` (pygame demo), `README.md`, `training_plot.png`,
`evaluation_results.csv`, `demo.mp4`.


## Setup

```bash
pip install numpy torch matplotlib   # required
pip install pygame                   # only for the visual demo (play.py)
```

- **Train:** `python train.py` (or the training cell below)
- **Evaluate trained vs random (10 episodes each):** `python evaluate.py`
- **Play yourself:** `python play.py human`
- **Watch the trained agent:** `python play.py agent`

Full instructions: `README.md` • Report: `report.md`


## 1. Game environment (`drone_env.py`)

2D map, two fixed circular obstacles (the big one blocks the straight path), delivery zone, momentum-based physics. Reward: **+100** delivery, **−100** crash, **−1**/step, **−25** timeout, plus potential-based shaping (0.5 × Δφ) rewarding distance progress and obstacle avoidance (Ng, Harada & Russell, 1999).

In [ ]:
"""
Drone Delivery -- Assignment 2 game environment.

A small 2D map with two fixed circular obstacles and one delivery target.
The drone has velocity (with drag) and must reach the target while avoiding
the obstacles. An episode ends when the drone reaches the target (success),
collides with an obstacle (crash), or a time limit is reached.

Pure-NumPy environment: no game engine is required for training.
An optional Pygame renderer is included for human play / agent demonstration.

Author: <Your Name>  |  Roll: <Your Roll>
Libraries used: numpy (and pygame only for the optional visualization).
"""

import numpy as np

# ----------------------------- game constants -------------------------------
WORLD_W = 600.0          # world width  (pixels)
WORLD_H = 400.0          # world height (pixels)

DRONE_R = 12.0           # drone radius
TARGET = (540.0, 200.0)  # delivery zone center
TARGET_R = 26.0          # delivery zone radius

# Two fixed obstacles (cx, cy, r) -- "two fixed obstacles" per the assignment.
# The first sits on the direct start->target line, so a straight dash is
# impossible: the drone must fly around it (and mind the second one).
OBSTACLES = ((300.0, 200.0, 55.0), (450.0, 290.0, 50.0))

START = (60.0, 200.0)    # episode start position
MAX_STEPS = 300          # time limit per episode

ACCEL = 300.0            # thrust acceleration (px/s^2)
DRAG = 0.92              # multiplicative velocity damping per step
MAX_SPEED = 250.0        # speed clamp (px/s)

DT = 1.0 / 30.0          # fixed physics step (30 steps per simulated second)

# Actions: 0=left, 1=right, 2=up, 3=down, 4=hover
ACTION_NAMES = ["LEFT", "RIGHT", "UP", "DOWN", "HOVER"]
N_ACTIONS = 5

# ------------------------------ reward values -------------------------------
# Simple numerical values, explained in the report:
R_SUCCESS = +100.0        # reaching the delivery zone
R_CRASH = -100.0          # colliding with an obstacle
R_STEP = -1.0             # per-step living cost (rewards finishing fast)
R_SHAPE = 0.5             # potential-based shaping scale (per potential unit)
SAFE_MARGIN = 25.0        # obstacle safety ring width used by the shaping
R_TIMEOUT = -25.0         # small extra penalty for running out of time

# ------------------------------ obs / state ---------------------------------
OBS_DIM = 6  # x, y, vx, vy, target dx (normalized), target dy (normalized)


def dist_to_target(x, y):
    return float(np.hypot(TARGET[0] - x, TARGET[1] - y))


def hits_obstacle(x, y):
    """Circle collision between drone and any obstacle."""
    for cx, cy, r in OBSTACLES:
        if np.hypot(x - cx, y - cy) < r + DRONE_R:
            return True
    return False


def in_target(x, y):
    return np.hypot(x - TARGET[0], y - TARGET[1]) <= TARGET_R - DRONE_R * 0.5


def potential(x, y):
    """Shaping potential phi(s):

    phi = -distance_to_target  -  total intrusion into obstacle safety rings.

    Potential-based shaping (Ng, Harada & Russell, 1999) adds
    R_shape = scale * (phi(s') - phi(s)) per step; it speeds up learning
    without changing the optimal policy. The safety-ring term makes the
    shaped reward prefer paths that go AROUND obstacles instead of the
    raw shortest line (which passes through the central obstacle).
    """
    phi = -dist_to_target(x, y)
    for cx, cy, r in OBSTACLES:
        gap = np.hypot(x - cx, y - cy) - (r + DRONE_R)
        if gap < SAFE_MARGIN:
            phi -= (SAFE_MARGIN - gap)  # 1 unit per pixel inside the ring
    return phi


class DroneDeliveryEnv:
    """
    Minimal 2D drone game with a NumPy-only reset()/step() interface.

    Observation (6 numbers, compact):
        0: x / WORLD_W
        1: y / WORLD_H
        2: vx / MAX_SPEED
        3: vy / MAX_SPEED
        4: (target_x - x) / WORLD_W
        5: (target_y - y) / WORLD_H
    """

    def __init__(self):
        self.reset()

    # ------------------------------------------------------------------ API
    def reset(self):
        self.x, self.y = START
        self.vx = self.vy = 0.0
        self.steps = 0
        self.score = 0.0          # visible game score (rewards collected)
        self.delivered = False
        self.crashed = False
        self._prev_pot = potential(self.x, self.y)
        return self._obs()

    def step(self, action):
        assert 0 <= action < N_ACTIONS

        # --- physics: thrust, drag, clamp, integrate --------------------
        ax = ay = 0.0
        if action == 0:
            ax = -ACCEL
        elif action == 1:
            ax = +ACCEL
        elif action == 2:
            ay = -ACCEL
        elif action == 3:
            ay = +ACCEL
        # action == 4: hover (no thrust)

        self.vx = (self.vx + ax * DT) * DRAG
        self.vy = (self.vy + ay * DT) * DRAG
        sp = float(np.hypot(self.vx, self.vy))
        if sp > MAX_SPEED:
            self.vx *= MAX_SPEED / sp
            self.vy *= MAX_SPEED / sp

        self.x += self.vx * DT
        self.y += self.vy * DT
        self.x = float(min(max(self.x, DRONE_R), WORLD_W - DRONE_R))
        self.y = float(min(max(self.y, DRONE_R), WORLD_H - DRONE_R))

        self.steps += 1
        reward = R_STEP
        terminated = False
        truncated = False

        # potential-based shaping: rewards getting closer to the target
        # AND escaping the obstacle safety rings (i.e., detouring around them)
        new_pot = potential(self.x, self.y)
        reward += (new_pot - self._prev_pot) * R_SHAPE
        self._prev_pot = new_pot

        # --- terminal checks -------------------------------------------
        if in_target(self.x, self.y):
            terminated = True
            self.delivered = True
            reward += R_SUCCESS
            self.score += 100
        elif hits_obstacle(self.x, self.y):
            terminated = True
            self.crashed = True
            reward += R_CRASH
        elif self.steps >= MAX_STEPS:
            truncated = True
            reward += R_TIMEOUT

        return self._obs(), float(reward), (terminated or truncated), {
            "delivered": self.delivered,
            "crashed": self.crashed,
            "score": self.score,
            "steps": self.steps,
        }

    def _obs(self):
        dx = (TARGET[0] - self.x) / WORLD_W
        dy = (TARGET[1] - self.y) / WORLD_H
        return np.array(
            [self.x / WORLD_W, self.y / WORLD_H,
             self.vx / MAX_SPEED, self.vy / MAX_SPEED, dx, dy],
            dtype=np.float32,
        )

    # --------------------------------------------------- human play (pygame)
    def render_init(self):
        import pygame
        pygame.init()
        self._screen = pygame.display.set_mode((int(WORLD_W), int(WORLD_H)))
        pygame.display.set_caption("Drone Delivery -- human play (arrow keys thrust)")
        self._clock = pygame.time.Clock()
        self._font = pygame.font.SysFont(None, 26)

    def render_frame(self):
        import pygame
        for event in pygame.event.get(pygame.QUIT):
            pygame.quit()
            raise SystemExit
        s = self._screen
        s.fill((18, 24, 38))
        # delivery zone
        pygame.draw.circle(s, (40, 120, 80), (int(TARGET[0]), int(TARGET[1])), int(TARGET_R))
        pygame.draw.circle(s, (70, 200, 130), (int(TARGET[0]), int(TARGET[1])), int(TARGET_R), 3)
        # obstacles
        for cx, cy, r in OBSTACLES:
            pygame.draw.circle(s, (120, 120, 130), (int(cx), int(cy)), int(r))
        # drone
        color = (90, 170, 255) if not self.crashed else (200, 60, 60)
        pygame.draw.circle(s, color, (int(self.x), int(self.y)), int(DRONE_R))
        pygame.draw.circle(s, (230, 240, 255), (int(self.x), int(self.y)), int(DRONE_R), 2)
        # HUD: visible score + step counter
        s.blit(self._font.render(
            f"Score: {self.score:.0f}   Steps: {self.steps}/{MAX_STEPS}", True, (235, 235, 235)),
            (12, 10))
        pygame.display.flip()
        self._clock.tick(30)

    def render_close(self):
        import pygame
        pygame.quit()


## 2. DQN agent (`dqn_agent.py`)

MLP Q-network 6→128→128→5, experience replay (50k), target network (sync every 1000 learn steps), ε-greedy with linear decay, Adam lr 1e-3, γ 0.99, batch 128, Huber loss.

In [ ]:
"""
DQN agent for Drone Delivery (Assignment 2).

A standard Deep Q-Network: MLP Q-function, experience replay, target network,
epsilon-greedy exploration with linear decay. The trained policy is saved as
state_dict + checkpoint so a play mode can reload it.

Author: <Your Name>  |  Roll: <Your Roll>
Libraries used: numpy, torch.
"""

import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn

import drone_env
from drone_env import DroneDeliveryEnv, N_ACTIONS, OBS_DIM

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class QNetwork(nn.Module):
    """Small MLP mapping the 6-d observation to one Q-value per action."""

    def __init__(self, obs_dim=OBS_DIM, n_actions=N_ACTIONS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=50_000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch):
        s, a, r, s2, d = zip(*random.sample(self.buf, batch))
        return (
            torch.as_tensor(np.array(s), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(a, dtype=torch.int64, device=DEVICE).unsqueeze(1),
            torch.as_tensor(r, dtype=torch.float32, device=DEVICE).unsqueeze(1),
            torch.as_tensor(np.array(s2), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(d, dtype=torch.float32, device=DEVICE).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buf)


class DQNAgent:
    def __init__(self, lr=1e-3, gamma=0.99, buffer_size=50_000, batch=128,
                 target_sync=1000):
        self.q = QNetwork().to(DEVICE)
        self.target = QNetwork().to(DEVICE)
        self.target.load_state_dict(self.q.state_dict())
        self.opt = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.gamma = gamma
        self.batch = batch
        self.target_sync = target_sync
        self.buffer = ReplayBuffer(buffer_size)
        self.learn_steps = 0

    # ------------------------------------------------------------- policy
    def act(self, obs, epsilon=0.0):
        if random.random() < epsilon:
            return random.randrange(N_ACTIONS)
        with torch.no_grad():
            t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            return int(self.q(t).argmax(dim=1).item())

    # ------------------------------------------------------------ learning
    def remember(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)

    def learn(self):
        if len(self.buffer) < self.batch:
            return None
        s, a, r, s2, d = self.buffer.sample(self.batch)
        with torch.no_grad():
            best_next = self.target(s2).max(dim=1, keepdim=True)[0]
            y = r + self.gamma * best_next * (1.0 - d)
        q_sa = self.q(s).gather(1, a)
        loss = nn.functional.smooth_l1_loss(q_sa, y)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()
        self.learn_steps += 1
        if self.learn_steps % self.target_sync == 0:
            self.target.load_state_dict(self.q.state_dict())
        return float(loss.item())

    # ------------------------------------------------------------ persist
    def save(self, path="dqn_drone.pth"):
        torch.save({
            "q_state": self.q.state_dict(),
            "obs_dim": OBS_DIM,
            "n_actions": N_ACTIONS,
        }, path)

    @classmethod
    def load(cls, path="dqn_drone.pth"):
        ckpt = torch.load(path, map_location=DEVICE)
        agent = cls()
        agent.q.load_state_dict(ckpt["q_state"])
        agent.target.load_state_dict(ckpt["q_state"])
        agent.q.eval()
        return agent


def random_action(_obs):
    return random.randrange(N_ACTIONS)


def run_episode(env, policy_fn, epsilon=0.0, max_steps=None, seed=None):
    """Run one full episode; returns (total_reward, info)."""
    if seed is not None:
        env.rng = np.random.default_rng(seed)
    obs = env.reset()
    total = 0.0
    steps = 0
    done = False
    while not done:
        a = policy_fn(obs) if epsilon == 0.0 else (
            random.randrange(N_ACTIONS) if random.random() < epsilon else policy_fn(obs))
        obs, r, done, info = env.step(a)
        total += r
        steps += 1
        if max_steps and steps >= max_steps:
            break
    return total, info


## 3. Training

1000 episodes, ε 1.0→0.05 over 700 episodes, seed 0. The best checkpoint (by 50-episode success rate) is kept. **Skip this cell if `dqn_drone.pth` already exists** — training takes a few minutes on CPU.

In [ ]:
"""
Train the DQN agent on Drone Delivery (Assignment 2).

Run:  python train.py

Produces:
    dqn_drone.pth      -- trained policy checkpoint
    training_plot.png  -- smoothed training reward per episode
    training_log.csv   -- raw per-episode rewards
"""

import csv
import io
import random

import numpy as np
import torch

from drone_env import DroneDeliveryEnv
from dqn_agent import DQNAgent

# ----------------------------- hyperparameters -------------------------------
EPISODES = 1000
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 700   # linear decay over episodes
GAMMA = 0.99
LR = 1e-3
BATCH = 128
BUFFER = 50_000
TARGET_SYNC = 1000        # learn steps between target-network syncs
LEARN_EVERY = 2           # gradient step every N env steps (CPU speed knob)
SEED = 0

SMOOTH = 30               # window for the smoothed plot line


def main():
    torch.set_num_threads(4)   # small nets train faster without huge thread pools
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    env = DroneDeliveryEnv()
    agent = DQNAgent(lr=LR, gamma=GAMMA, buffer_size=BUFFER, batch=BATCH,
                     target_sync=TARGET_SYNC)

    rewards, wins = [], []
    best_sr = -1.0

    for ep in range(EPISODES):
        eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / EPS_DECAY)
        obs = env.reset()
        total = 0.0
        done = False
        step_i = 0
        while not done:
            a = agent.act(obs, epsilon=eps)
            nobs, r, done, info = env.step(a)
            agent.remember(obs, a, r, nobs, float(done))
            obs = nobs
            total += r
            if step_i % LEARN_EVERY == 0:
                agent.learn()   # no-op until the buffer holds at least one batch
            step_i += 1
        rewards.append(total)
        wins.append(1.0 if info["delivered"] else 0.0)

        if (ep + 1) % 50 == 0:
            m = np.mean(rewards[-50:])
            sr = np.mean(wins[-50:])
            print(f"episode {ep+1:5d} | eps {eps:.2f} | "
                  f"reward(avg50) {m:8.1f} | success(avg50) {sr:.2f}", flush=True)
            # keep the best checkpoint (DQN can drift late in training)
            if sr > best_sr:
                best_sr = sr
                agent.save("dqn_drone.pth")
                print(f"          new best policy (success {sr:.2f}) saved", flush=True)

    agent.save("dqn_drone_last.pth")
    print(f"saved final policy -> dqn_drone_last.pth (best avg50 success: {best_sr:.2f})")
    print("best policy kept in  -> dqn_drone.pth")

    # ------------------------------ plot --------------------------------
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    raw = np.array(rewards)
    smooth = np.convolve(raw, np.ones(SMOOTH) / SMOOTH, mode="valid")
    plt.figure(figsize=(9, 5))
    plt.plot(raw, alpha=0.25, label="per-episode reward")
    plt.plot(range(SMOOTH - 1, len(raw)), smooth, label=f"{SMOOTH}-episode mean")
    plt.xlabel("Training episode")
    plt.ylabel("Episode reward")
    plt.title("DQN training reward -- Drone Delivery")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    # write via BytesIO: matplotlib's PIL save can fail on OneDrive-synced folders
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150)
    with open("training_plot.png", "wb") as f:
        f.write(buf.getvalue())
    print("saved plot -> training_plot.png")

    with open("training_log.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["episode", "reward"])
        for i, r in enumerate(rewards, 1):
            w.writerow([i, f"{r:.2f}"])
    print("saved log -> training_log.csv")

    # quick greedy sanity check with the saved best policy
    best = DQNAgent.load("dqn_drone.pth")
    d = 0
    for seed in range(10):
        random.seed(seed)
        obs = env.reset()
        done = False
        while not done:
            obs, r, done, info = env.step(best.act(obs, epsilon=0.0))
        d += int(info["delivered"])
    print(f"greedy check with best policy: {d}/10 deliveries")



if __name__ == "__main__":
    main()


## 4. Training curve

Success reaches 1.00 (50-ep avg) around episode 750; the best checkpoint is from that stable peak.

In [ ]:
import matplotlib.pyplot as plt
import csv, numpy as np

raw = np.array([float(r[1]) for r in list(csv.reader(open("training_log.csv")))[1:]])
smooth = np.convolve(raw, np.ones(30)/30, mode="valid")
plt.figure(figsize=(9,5))
plt.plot(raw, alpha=0.25, label="per-episode reward")
plt.plot(range(29, len(raw)), smooth, label="30-episode mean")
plt.xlabel("Training episode"); plt.ylabel("Episode reward")
plt.title("DQN training reward -- Drone Delivery"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("training_plot.png", dpi=150); plt.show()

## 5. Evaluation: trained vs random (10 episodes, same seeds)

Trained: **+148.1 avg reward, 10/10 deliveries, 0 crashes**. Random: **−332.6, 0/10** (always times out).

In [ ]:
"""
Evaluate trained vs random agent on Drone Delivery (Assignment 2).

Runs 10 episodes with the trained policy and 10 with a random policy on
identical starting scenarios (same env, deterministic start; the seed makes
any stochasticity match). Reports average episode reward, deliveries out of
10, crashes out of 10 and average steps, and writes evaluation_results.csv.

Run:  python evaluate.py
"""

import csv
import random

import numpy as np

from drone_env import DroneDeliveryEnv
from dqn_agent import DQNAgent

N_EPISODES = 10
SEEDS = list(range(10))          # same starting scenarios for both agents


def run_eval(policy_fn, label):
    env = DroneDeliveryEnv()
    rewards, deliveries, crashes, steps_list = [], 0, 0, []
    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        env.reset()
        obs = env.reset()
        total, steps, done = 0.0, 0, False
        while not done:
            a = policy_fn(obs)
            obs, r, done, info = env.step(a)
            total += r
            steps += 1
        rewards.append(total)
        deliveries += int(info["delivered"])
        crashes += int(info["crashed"])
        steps_list.append(steps)
    print(f"\n=== {label} ({N_EPISODES} episodes) ===")
    print(f"avg episode reward : {np.mean(rewards):8.1f}")
    print(f"deliveries         : {deliveries}/10")
    print(f"crashes            : {crashes}/10")
    print(f"avg steps          : {np.mean(steps_list):6.1f}")
    return {
        "agent": label,
        "avg_reward": round(float(np.mean(rewards)), 2),
        "deliveries": deliveries,
        "crashes": crashes,
        "avg_steps": round(float(np.mean(steps_list)), 1),
        "rewards": [round(r, 2) for r in rewards],
    }


def main():
    agent = DQNAgent.load("dqn_drone.pth")
    trained = run_eval(lambda o: agent.act(o, epsilon=0.0), "Trained DQN agent")
    rnd = run_eval(lambda o: random.randrange(5), "Random agent")

    with open("evaluation_results.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["agent", "avg_reward", "deliveries_out_of_10",
                    "crashes_out_of_10", "avg_steps"])
        for row in (trained, rnd):
            w.writerow([row["agent"], row["avg_reward"], row["deliveries"],
                        row["crashes"], row["avg_steps"]])
        w.writerow([])
        w.writerow(["per_episode_rewards_trained"] + trained["rewards"])
        w.writerow(["per_episode_rewards_random"] + rnd["rewards"])
    print("\nsaved -> evaluation_results.csv")


if __name__ == "__main__":
    main()


## 6. Visual demo (needs pygame)

Run locally in a terminal (not inside this notebook):

```bash
python play.py human   # you fly with arrow keys
python play.py agent   # saved DQN policy flies
```

`demo.mp4` in the submission shows both segments recorded.